# Real Databases

Server-side validation against live database engines using [Testcontainers](https://testcontainers.com/). **Requires Docker.**

- [9. Server-Side Database Validation](#1-server-side-database-validation) — PostgreSQL, MySQL, PySpark, DuckDB ATTACH

> Builds on the [Core Tutorial](../1_core_tutorial/core_tutorial.ipynb). Each section spins up its own container and tears it down at the end.

## Setup

This notebook re-resolves the dataset paths so it runs on its own. The database sections below import everything else they need.

In [ ]:
# Install vowl (skipped during execution)
# %pip install 'vowl[all]'

In [ ]:
from pathlib import Path

# Walk up to the repo root (works no matter how deep this notebook sits)
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "tests" / "hdb_resale").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

# Single-source: HDB Resale dataset
HDB_DIR = REPO_ROOT / "tests" / "hdb_resale"
HDB_CSV = HDB_DIR / "HDBResaleWithErrors.csv"
HDB_CONTRACT = HDB_DIR / "hdb_resale_simple.yaml"  # Switch to "hdb_resale.yaml" for a more complex contract

# Multi-source: Employee dataset (payroll + employee list)
EMPLOYEE_DIR = REPO_ROOT / "tests" / "employee"
EMPLOYEE_PAYROLL_CSV = EMPLOYEE_DIR / "demo_employee_payroll.csv"
EMPLOYEE_LIST_CSV = EMPLOYEE_DIR / "demo_employee_list.csv"
EMPLOYEE_CONTRACT = EMPLOYEE_DIR / "employee_payroll_datacontract.yaml"

# Common imports used throughout this notebook
import pandas as pd
from vowl import validate_data

# --- Quieten known, benign warnings emitted by this demo ---
import warnings

# vowl surfaces UserWarnings (Arrow type coercion, multi-schema adapter reuse)
# that are informational for this demo dataset.
warnings.filterwarnings("ignore", category=UserWarning,
                        module=r"vowl\.validation\.runner")

The examples below use [testcontainers](https://testcontainers-python.readthedocs.io/) to spin up real database instances in Docker. Unlike the Core Tutorial, where checks run client-side in DuckDB, here the SQL checks execute **on the database server itself** (PostgreSQL, MySQL, PySpark, etc.).

**Requirements:**

- Docker must be running
- Dev dependencies: `uv sync --dev`

<a id="1-server-side"></a>
## 1. Server-Side Database Validation

### 1.1 PostgreSQL

In [46]:
import os
from pathlib import Path as _Path

import ibis
import pandas as pd
from testcontainers.postgres import PostgresContainer
from vowl import validate_data
from vowl.adapters import IbisAdapter

# --- Docker / Infrastructure Setup ---
# Configure Docker Desktop socket path (macOS)
docker_sock = _Path.home() / ".docker" / "run" / "docker.sock"
if docker_sock.exists() and "DOCKER_HOST" not in os.environ:
    os.environ["DOCKER_HOST"] = f"unix://{docker_sock}"

# Disable Ryuk reaper for Docker Desktop compatibility
os.environ.setdefault("TESTCONTAINERS_RYUK_DISABLED", "true")

postgres = PostgresContainer("postgres:15-alpine")
postgres.start()

# Connect via Ibis
pg_con = ibis.postgres.connect(
    host=postgres.get_container_host_ip(),
    port=postgres.get_exposed_port(5432),
    user=postgres.username,
    password=postgres.password,
    database=postgres.dbname,
)

# Load HDB data as TEXT for PostgreSQL
hdb_df = pd.read_csv(HDB_CSV, low_memory=False).fillna("").astype(str)

pg_con.raw_sql("""
    CREATE TABLE hdb_resale_prices (
        month TEXT, town TEXT, flat_type TEXT, block TEXT,
        street_name TEXT, storey_range TEXT, floor_area_sqm TEXT,
        flat_model TEXT, lease_commence_date TEXT, remaining_lease TEXT,
        resale_price TEXT
    )
""")
pg_con.insert("hdb_resale_prices", hdb_df)
print("PostgreSQL container started and data loaded.")

PostgreSQL container started and data loaded.


In [47]:
# --- Validation (runs server-side on PostgreSQL) ---
adapter = IbisAdapter(pg_con)
result = validate_data(contract=str(HDB_CONTRACT), adapter=adapter)

In [48]:
# Show the report, then tear down the container
result.display_full_report()
postgres.stop()



=== Data Quality Validation Results ===
   Contract Version:      v3.1.0
   Contract ID:           c11443ee-542f-4442-b28d-2d224342be37
   Schemas:               hdb_resale_prices

 OVERALL DATA QUALITY
   Overall:
     Checks Pass Rate:       15 / 20 (75.0%)
     ERRORED Checks:         2

   hdb_resale_prices:
     Overall:
       Checks Pass Rate:       15 / 20 (75.0%)
       ERRORED Checks:         2
     Single Table:
       Checks Pass Rate:       15 / 20 (75.0%)
       ERRORED Checks:         2
       Unique Passed Rows:     201,873 / 201,879 (99.9%)
     Multi Table:
       Checks Pass Rate:       0 / 0 (N/A)
       ERRORED Checks:         0
       Non-unique Failed Rows: 0


 CHECK RESULTS
+-----------------------------------------+---------------------------------------+-------------------+--------+---------------+---------------+--------+----------------+
| check_id                                | Target                                | tables_in_query   | status | operat

> **Why do 2 checks ERROR on PostgreSQL but not on MySQL?**
>
> Both backends store all columns as TEXT, but the checks `floor_area_must_be_less_than_200` and `resale_price_must_not_exceed_2m` compare columns to numeric literals (e.g. `WHERE floor_area_sqm >= 200`). vowl's `apply_try_cast` transform wraps the column in a safe cast, but the rendered SQL differs by backend:
>
> | Backend | Rendered SQL | Behaviour |
> |---------|-------------|-----------|
> | **MySQL** | `CAST(floor_area_sqm AS SIGNED) >= 200` | MySQL truncates `"130.0"` to `130` -- no error |
> | **PostgreSQL** | `CAST(floor_area_sqm AS BIGINT) >= 200` | PostgreSQL has no `TRY_CAST`, so sqlglot downgrades to strict `CAST`, which rejects `"130.0"` as invalid integer syntax |
>
> In a production PostgreSQL setup, you would use proper column types (`INTEGER`, `NUMERIC`), which would:
> 1. **Reject invalid data at INSERT time** -- values like `"abc"` or `""` would never enter the table
> 2. **Make the CAST unnecessary** -- the column is already numeric, so `WHERE floor_area_sqm >= 200` executes directly
>
> See [Known Issues](../../docs/known-issues.md) for more on backend-specific dialect differences.

### 1.2 MySQL

> **Note:** For MySQL, select the database when creating the connection (via `database=` param or in the connection URI). vowl does not issue `USE database`; it runs read-only `SELECT` queries against the active database.

In [49]:
from testcontainers.mysql import MySqlContainer

# --- Docker / Infrastructure Setup ---
mysql = MySqlContainer("mysql:8.0")
mysql.start()

mysql_con = ibis.mysql.connect(
    host=mysql.get_container_host_ip(),
    port=int(mysql.get_exposed_port(3306)),
    user=mysql.username,
    password=mysql.password,
    database=mysql.dbname,
)

# Load data. MySQL requires TEXT types for string columns
hdb_df = pd.read_csv(HDB_CSV, low_memory=False).fillna("").astype(str)

mysql_con.raw_sql("""
    CREATE TABLE hdb_resale_prices (
        month TEXT, town TEXT, flat_type TEXT, block TEXT,
        street_name TEXT, storey_range TEXT, floor_area_sqm TEXT,
        flat_model TEXT, lease_commence_date TEXT, remaining_lease TEXT,
        resale_price TEXT
    )
""")
mysql_con.insert("hdb_resale_prices", hdb_df)
print("MySQL container started and data loaded.")

MySQL container started and data loaded.


In [50]:
# --- Validation (runs server-side on MySQL) ---
adapter = IbisAdapter(mysql_con)
result = validate_data(contract=str(HDB_CONTRACT), adapter=adapter)

In [51]:
# Show the report, then tear down the container
result.display_full_report()
mysql.stop()



=== Data Quality Validation Results ===
   Contract Version:      v3.1.0
   Contract ID:           c11443ee-542f-4442-b28d-2d224342be37
   Schemas:               hdb_resale_prices

 OVERALL DATA QUALITY
   Overall:
     Checks Pass Rate:       16 / 20 (80.0%)

   hdb_resale_prices:
     Overall:
       Checks Pass Rate:       16 / 20 (80.0%)
       ERRORED Checks:         0
     Single Table:
       Checks Pass Rate:       16 / 20 (80.0%)
       ERRORED Checks:         0
       Unique Passed Rows:     201,861 / 201,879 (99.9%)
     Multi Table:
       Checks Pass Rate:       0 / 0 (N/A)
       ERRORED Checks:         0
       Non-unique Failed Rows: 0


 CHECK RESULTS
+-----------------------------------------+---------------------------------------+-------------------+--------+---------------+---------------+--------+----------------+
| check_id                                | Target                                | tables_in_query   | status | operator      | expected      | actua

### 1.3 PySpark

PySpark validation uses `ibis.pyspark.connect()` to bridge Spark SQL with vowl's `IbisAdapter`. The SQL checks execute within the Spark engine.

**Requirements:**

- PySpark installed (`pip install pyspark`)
- Java 11+ available (set `JAVA_HOME` if needed)

In [52]:
import os

# Auto-detect Java on macOS (Homebrew) if JAVA_HOME not set
if "JAVA_HOME" not in os.environ:
    from pathlib import Path as _P
    homebrew_java = _P("/opt/homebrew/opt/openjdk@17")
    if homebrew_java.exists():
        os.environ["JAVA_HOME"] = str(homebrew_java)

from pyspark.sql import SparkSession

# --- Spark Setup ---
spark = (
    SparkSession.builder
    .master("local[1]")
    .appName("vowl_demo")
    .config("spark.driver.memory", "512m")
    .config("spark.sql.shuffle.partitions", "1")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

# Load the data as a Spark DataFrame
spark_df = spark.createDataFrame(
    pd.read_csv(HDB_CSV, low_memory=False).fillna("").astype(str)
)
spark_df.createOrReplaceTempView("hdb_resale_prices")

# --- Validation (runs within the Spark engine) ---
con = ibis.pyspark.connect(session=spark)
adapter = IbisAdapter(con)
result = validate_data(contract=str(HDB_CONTRACT), adapter=adapter)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/06/28 22:05:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Exception ignored while finalizing file <_io.BufferedWriter name=5>:
Traceback (most recent call last):
  File "/Users/jamestth/DEP/DCP/dqmk/.venv/lib/python3.14/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 200, in manager
BrokenPipeError: [Errno 32] Broken pipe


In [53]:
# Show the report, then stop the Spark session
result.display_full_report()
spark.stop()



=== Data Quality Validation Results ===
   Contract Version:      v3.1.0
   Contract ID:           c11443ee-542f-4442-b28d-2d224342be37
   Schemas:               hdb_resale_prices

 OVERALL DATA QUALITY
   Overall:
     Checks Pass Rate:       17 / 20 (85.0%)

   hdb_resale_prices:
     Overall:
       Checks Pass Rate:       17 / 20 (85.0%)
       ERRORED Checks:         0
     Single Table:
       Checks Pass Rate:       17 / 20 (85.0%)
       ERRORED Checks:         0
       Unique Passed Rows:     201,873 / 201,879 (99.9%)
     Multi Table:
       Checks Pass Rate:       0 / 0 (N/A)
       ERRORED Checks:         0
       Non-unique Failed Rows: 0


 CHECK RESULTS
+-----------------------------------------+---------------------------------------+-------------------+--------+---------------+---------------+--------+----------------+
| check_id                                | Target                                | tables_in_query   | status | operator      | expected      | actua

### 1.4 DuckDB ATTACH (Multi-Source Workaround)

As shown in 1.1 (PostgreSQL), some backends produce `ERROR` checks due to dialect limitations (e.g. PostgreSQL lacks `TRY_CAST`, MSSQL lacks regex). A workaround is to **attach** the remote database to a local DuckDB instance and run all queries through DuckDB's engine instead.

This gives you DuckDB's full SQL support (`TRY_CAST`, `REGEXP_MATCHES`, etc.) while reading data directly from the remote database -- no manual data loading required. Here we use the Employee dataset to demonstrate cross-table validation through DuckDB ATTACH.

> **Note:** DuckDB ATTACH currently supports PostgreSQL, MySQL, and SQLite. See [Known Issues](../../docs/known-issues.md#mssql-no-regex-support) for the MSSQL workaround using the same pattern.

In [54]:
# --- Simulate two separate source systems, each with its own database ---
pg_payroll = PostgresContainer("postgres:15-alpine")
pg_payroll.start()

pg_employees = PostgresContainer("postgres:15-alpine")
pg_employees.start()

# Load payroll data into the first database
payroll_con = ibis.postgres.connect(
    host=pg_payroll.get_container_host_ip(),
    port=pg_payroll.get_exposed_port(5432),
    user=pg_payroll.username, password=pg_payroll.password, database=pg_payroll.dbname,
)
payroll_con.create_table("demo_employee_payroll", pd.read_csv(EMPLOYEE_PAYROLL_CSV).astype(str))

# Load employee list into the second database
employees_con = ibis.postgres.connect(
    host=pg_employees.get_container_host_ip(),
    port=pg_employees.get_exposed_port(5432),
    user=pg_employees.username, password=pg_employees.password, database=pg_employees.dbname,
)
employees_con.create_table("demo_employee_list", pd.read_csv(EMPLOYEE_LIST_CSV).astype(str))
print("Two PostgreSQL containers started (payroll + employee list).")

# --- DuckDB ATTACH: bridge both databases into one DuckDB connection ---
duck_con = ibis.duckdb.connect()
duck_con.raw_sql(f"""
    ATTACH 'dbname={pg_payroll.dbname} host={pg_payroll.get_container_host_ip()}
           port={pg_payroll.get_exposed_port(5432)}
           user={pg_payroll.username} password={pg_payroll.password}'
    AS payroll_db (TYPE postgres, READ_ONLY)
""")
duck_con.raw_sql(f"""
    ATTACH 'dbname={pg_employees.dbname} host={pg_employees.get_container_host_ip()}
           port={pg_employees.get_exposed_port(5432)}
           user={pg_employees.username} password={pg_employees.password}'
    AS employees_db (TYPE postgres, READ_ONLY)
""")

# Attached tables are namespaced (e.g. payroll_db.public.demo_employee_payroll).
# Create views as prefix-free shortcuts so contract queries can reference tables directly.
duck_con.raw_sql("USE memory")
duck_con.raw_sql("CREATE VIEW demo_employee_payroll AS SELECT * FROM payroll_db.public.demo_employee_payroll")
duck_con.raw_sql("CREATE VIEW demo_employee_list AS SELECT * FROM employees_db.public.demo_employee_list")

# Validate -- cross-table checks (e.g. referential integrity between payroll
# and employee list) work because both views are on the same DuckDB connection.
adapter = IbisAdapter(duck_con)
result = validate_data(contract=str(EMPLOYEE_CONTRACT), adapter=adapter)

Two PostgreSQL containers started (payroll + employee list).


In [55]:
# Show the report, then tear down both containers
result.display_full_report()
pg_payroll.stop()
pg_employees.stop()



=== Data Quality Validation Results ===
   Contract Version:      v3.1.0
   Contract ID:           b6376e57-aa90-41eb-90d0-c72dca7567e6
   Schemas:               demo_employee_payroll, demo_employee_list

 OVERALL DATA QUALITY
   Overall:
     Checks Pass Rate:       68 / 76 (89.4%)
     ERRORED Checks:         2

   demo_employee_payroll:
     Overall:
       Checks Pass Rate:       46 / 53 (86.7%)
       ERRORED Checks:         2
     Single Table:
       Checks Pass Rate:       46 / 51 (90.1%)
       ERRORED Checks:         2
       Unique Passed Rows:     0 / 3 (0.0%)
     Multi Table:
       Checks Pass Rate:       0 / 2 (0.0%)
       ERRORED Checks:         0
       Non-unique Failed Rows: 3

   demo_employee_list:
     Overall:
       Checks Pass Rate:       22 / 23 (95.6%)
       ERRORED Checks:         0
     Single Table:
       Checks Pass Rate:       22 / 23 (95.6%)
       ERRORED Checks:         0
       Unique Passed Rows:     0 / 2 (0.0%)
     Multi Table:
       Check